## 05 Barrier Layers

**笔记本**：`05 barrier_layers20260403.1.ipynb`

**库**：`geopandas`、`pandas`、`numpy`、`osmium`（pyosmium）、`matplotlib`、`shapely`（`Point`、`LineString`、`Polygon`、`MultiPolygon`、`prepared`、`unary_union`）等。

**输入**：
- `data_raw/guangdong.osm.pbf`
- `data_raw/shenzhen_boundary.geojson`
- `../04 Transport/data_out/sz_drive_edges.gpkg`（高速/主干、桥、隧筛选）
- `../00/04-transport-download-data/1-road-network-rail/shenzhen_railway_opensource.shp`（地面铁路补充，可选）

**做了什么 / 算了什么**：
读深圳边界与 bbox，先有深圳 polygon，再有一个 bbox 作为快速粗筛范围
用 pyosmium 提取面状水体：natural=water / wetland、landuse=reservoir/basin 等 
线状水系：waterway ∈ river/stream/canal/drain/ditch，裁剪。
铁路：PBF 中 railway 若干类，去掉 tunnel=yes，保留会在地表形成横向割裂的铁路；与 00 文件夹铁路 SHP 合并、去重（按 name、railway 等），叠加了一份百度网盘铁路矢量数据。
从已有车行道路里再派生几类“交通障碍/边界”：从 04 的  sz_drive_edges  筛选：高等级道路（motorway/trunk 及其 link）、桥梁（bridge=yes）、隧道（tunnel=yes）。
输出成 GPKG；再为每类障碍做 unary_union，写入 sz_barrier_unions.gpkg，方便下游做相交判断。
本笔记本里不对这些几何做固定米级 buffer（以原始几何与合并为主），没有额外人为扩成“影响带”（没有做buffer）。

**写出文件**（均在 `data_out/`）：
- `sz_barrier_water.gpkg`、`sz_barrier_waterway.gpkg`、`sz_barrier_railway.gpkg`
- `sz_barrier_highway_major.gpkg`、`sz_barrier_bridges.gpkg`、`sz_barrier_tunnels.gpkg`
- `sz_barrier_unions.gpkg`（按 `barrier_type` 一行一类 union 几何）

**典型输出信息**：各图层要素数量、`total edges` / `highway_major` / `bridges` / `tunnels` 统计、`unions saved` 与 `Done.`。
障碍物空间图层；统一合并后的障碍几何；供后续相交/穿越/邻近分析用的 barrier 基础库
支持下游分析：判断两点之间是否有障碍横亘； 识别“为什么会绕行”/绕行是被什么结构逼出来的；作为构建 Ground Friction 指标； 找“空中有优势”的空间


In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import osmium
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
from shapely.prepared import prep as shapely_prep
from shapely.ops import unary_union
import warnings
warnings.filterwarnings("ignore")

RAW = Path("data_raw")
OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

PBF = RAW / "guangdong.osm.pbf"
BOUNDARY = RAW / "shenzhen_boundary.geojson"
EDGES_04 = Path("../04 Transport/data_out/sz_drive_edges.gpkg")

# 00 补充数据 (铁路线 — 比 OSM 更全)
RAILWAY_00 = Path("../00/04-transport-download-data/1-road-network-rail/shenzhen_railway_opensource.shp")

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
sz_prep = shapely_prep(shenzhen_geom)
bbox = shenzhen_geom.bounds

print(f"Boundary: {shenzhen_geom.geom_type}")
print(f"PBF: {PBF.exists()}")
print(f"04 edges: {EDGES_04.exists()}")
print(f"00 铁路: {RAILWAY_00.exists()}")

Boundary: MultiPolygon
PBF: True
04 edges: True
00 铁路: True


In [2]:
# ============================================================
#  1. 水体 (natural=water)
#  用 pyosmium 从 PBF 提取, 修复 GDAL multipolygons 层返回 0 条的问题
#  同时提取 landuse=reservoir 作为补充
# ============================================================

class WaterHandler(osmium.SimpleHandler):
    """提取 natural=water 和 landuse=reservoir 的闭合 ways."""

    WATER_TAGS = {
        ("natural", "water"),
        ("natural", "wetland"),
        ("landuse", "reservoir"),
        ("landuse", "basin"),
    }

    def __init__(self, minx, miny, maxx, maxy):
        super().__init__()
        self.minx, self.miny, self.maxx, self.maxy = minx, miny, maxx, maxy
        self.polygons = []

    def area(self, a):
        tags = dict(a.tags)
        if not any((k, tags.get(k)) in self.WATER_TAGS for k, _ in self.WATER_TAGS):
            return
        try:
            outer = a.outer_rings()
        except Exception:
            return
        for ring in outer:
            coords = [(n.lon, n.lat) for n in ring if n.location.valid()]
            if len(coords) < 4:
                continue
            lons = [c[0] for c in coords]
            lats = [c[1] for c in coords]
            if max(lons) < self.minx or min(lons) > self.maxx:
                continue
            if max(lats) < self.miny or min(lats) > self.maxy:
                continue
            try:
                poly = Polygon(coords)
                if poly.is_valid and poly.area > 0:
                    self.polygons.append({
                        "natural": tags.get("natural", ""),
                        "landuse": tags.get("landuse", ""),
                        "name": tags.get("name", ""),
                        "geometry": poly,
                    })
            except Exception:
                pass

print("Extracting water bodies from PBF ...")
wh = WaterHandler(*bbox)
wh.apply_file(str(PBF), locations=True)
print(f"  bbox-filtered: {len(wh.polygons)} polygons")

if wh.polygons:
    water = gpd.GeoDataFrame(wh.polygons, crs=4326)
    water = gpd.clip(water, shenzhen_geom)
else:
    water = gpd.GeoDataFrame(columns=["natural", "landuse", "name", "geometry"],
                              geometry="geometry", crs=4326)
print(f"  clipped: {len(water)} water features")
del wh

Extracting water bodies from PBF ...
  bbox-filtered: 5724 polygons
  clipped: 2161 water features


In [3]:
# ============================================================
#  2. 水系 (waterway=river/stream/canal/drain/ditch)
#  用 pyosmium 从 PBF 提取线状水系
# ============================================================

WATERWAY_TYPES = {"river", "stream", "canal", "drain", "ditch"}

class WaterwayHandler(osmium.SimpleHandler):
    def __init__(self, minx, miny, maxx, maxy):
        super().__init__()
        self.minx, self.miny, self.maxx, self.maxy = minx, miny, maxx, maxy
        self.features = []

    def way(self, w):
        ww = w.tags.get("waterway")
        if ww not in WATERWAY_TYPES:
            return
        pts = [(n.location.lon, n.location.lat) for n in w.nodes if n.location.valid()]
        if len(pts) < 2:
            return
        lons = [p[0] for p in pts]
        lats = [p[1] for p in pts]
        if max(lons) < self.minx or min(lons) > self.maxx:
            return
        if max(lats) < self.miny or min(lats) > self.maxy:
            return
        tags = dict(w.tags)
        self.features.append({
            "waterway": ww,
            "name": tags.get("name", ""),
            "geometry": LineString(pts),
        })

print("Extracting waterways from PBF ...")
wwh = WaterwayHandler(*bbox)
wwh.apply_file(str(PBF), locations=True)
print(f"  bbox-filtered: {len(wwh.features)} lines")

if wwh.features:
    waterway = gpd.GeoDataFrame(wwh.features, crs=4326)
    waterway = gpd.clip(waterway, shenzhen_geom)
else:
    waterway = gpd.GeoDataFrame(columns=["waterway", "name", "geometry"],
                                 geometry="geometry", crs=4326)
print(f"  clipped: {len(waterway)} waterway features")
del wwh

Extracting waterways from PBF ...
  bbox-filtered: 6517 lines
  clipped: 2061 waterway features


In [4]:
# ============================================================
#  3. 铁路 (railway=rail/light_rail/tram/narrow_gauge)
#  用 pyosmium 精确提取, 修复 GDAL other_tags 只返回 1 条的问题
#  排除 subway (地下, 不构成地面障碍)
# ============================================================

RAILWAY_SURFACE = {"rail", "light_rail", "tram", "narrow_gauge", "construction"}

class RailwayHandler(osmium.SimpleHandler):
    def __init__(self, minx, miny, maxx, maxy):
        super().__init__()
        self.minx, self.miny, self.maxx, self.maxy = minx, miny, maxx, maxy
        self.features = []

    def way(self, w):
        rw = w.tags.get("railway")
        if rw not in RAILWAY_SURFACE:
            return
        tags = dict(w.tags)
        if tags.get("tunnel") == "yes":
            return
        pts = [(n.location.lon, n.location.lat) for n in w.nodes if n.location.valid()]
        if len(pts) < 2:
            return
        lons = [p[0] for p in pts]
        lats = [p[1] for p in pts]
        if max(lons) < self.minx or min(lons) > self.maxx:
            return
        if max(lats) < self.miny or min(lats) > self.maxy:
            return
        self.features.append({
            "railway": rw,
            "name": tags.get("name", ""),
            "bridge": tags.get("bridge", "no"),
            "geometry": LineString(pts),
        })

print("Extracting railways from PBF ...")
rh = RailwayHandler(*bbox)
rh.apply_file(str(PBF), locations=True)
print(f"  bbox-filtered: {len(rh.features)} segments")

if rh.features:
    railway = gpd.GeoDataFrame(rh.features, crs=4326)
    railway = gpd.clip(railway, shenzhen_geom)
else:
    railway = gpd.GeoDataFrame(columns=["railway", "name", "bridge", "geometry"],
                                geometry="geometry", crs=4326)
print(f"  clipped (OSM): {len(railway)} railway features")

# ── 合并 00 铁路数据 (2,451 LineString, 比 OSM 更全) ──
if RAILWAY_00.exists():
    rw00 = gpd.read_file(RAILWAY_00).to_crs(4326)
    rw00 = gpd.clip(rw00, shenzhen_geom)
    # 只保留地面铁路 (排除 tunnel=yes)
    if "tunnel" in rw00.columns:
        rw00 = rw00[rw00["tunnel"] != "yes"]
    # 统一字段名
    rw00_std = gpd.GeoDataFrame({
        "railway": rw00["fclass"] if "fclass" in rw00.columns else "rail",
        "name": rw00["name"] if "name" in rw00.columns else "",
        "bridge": rw00["bridge"] if "bridge" in rw00.columns else "no",
        "geometry": rw00.geometry,
    }, crs=4326)
    print(f"  00 铁路: {len(rw00_std)} features")

    # 合并 + 去重 (按几何近似去重: buffer 10m 后 dissolve 不合适, 直接 concat)
    railway = pd.concat([railway, rw00_std], ignore_index=True)
    railway = railway.drop_duplicates(subset=["name", "railway"], keep="first")
    print(f"  merged + dedup: {len(railway)} railway features")
    del rw00, rw00_std
else:
    print("  00 铁路 not found, using OSM only")

del rh

Extracting railways from PBF ...
  bbox-filtered: 2102 segments
  clipped (OSM): 1209 railway features
  00 铁路: 2451 features
  merged + dedup: 70 railway features


In [5]:
# ============================================================
#  4. 高速/主干路 + 桥梁 + 隧道
#  直接读取 04 Transport 输出的 edges GeoPackage, 按属性筛选
# ============================================================

print(f"Loading edges from {EDGES_04} ...")
edges = gpd.read_file(EDGES_04)
print(f"  total edges: {len(edges):,}")

def _match(val, targets):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return False
    return any(v in targets for v in (val if isinstance(val, list) else [val]))

# 高速 / 主干路
major_types = ["motorway", "motorway_link", "trunk", "trunk_link"]
highways_barrier = edges[edges["highway"].apply(lambda x: _match(x, major_types))].copy()
print(f"  highway_major: {len(highways_barrier):,}")

# 桥梁 / 高架
bridges = edges[edges["bridge"] == "yes"].copy() if "bridge" in edges.columns else gpd.GeoDataFrame()
print(f"  bridges: {len(bridges):,}")

# 隧道
tunnels = edges[edges["tunnel"] == "yes"].copy() if "tunnel" in edges.columns else gpd.GeoDataFrame()
print(f"  tunnels: {len(tunnels):,}")

del edges

Loading edges from ../04 Transport/data_out/sz_drive_edges.gpkg ...
  total edges: 661,793
  highway_major: 42,016
  bridges: 22,147
  tunnels: 3,447


In [6]:
# ============================================================
#  5. 保存所有 barrier 图层 + 生成 union 几何 (加速下游空间查询)
# ============================================================

barrier_layers = {
    "water": water,
    "waterway": waterway,
    "railway": railway,
    "highway_major": highways_barrier,
    "bridges": bridges,
    "tunnels": tunnels,
}

print("=== Barrier Layers Summary ===")
for name, gdf in barrier_layers.items():
    n = len(gdf)
    if n > 0:
        gdf.to_file(OUT / f"sz_barrier_{name}.gpkg", driver="GPKG")
    print(f"  {name:20s} {n:>6,} features {'[saved]' if n else '[empty]'}")

# 生成 union 几何, 用于 10 OD Analysis 里快速 intersects 判定
unions = {}
for name, gdf in barrier_layers.items():
    if len(gdf) > 0:
        unions[name] = unary_union(gdf.geometry)
    else:
        unions[name] = None

union_rows = []
for name, geom in unions.items():
    if geom is not None:
        union_rows.append({"barrier_type": name, "geometry": geom})

if union_rows:
    unions_gdf = gpd.GeoDataFrame(union_rows, crs=4326)
    unions_gdf.to_file(OUT / "sz_barrier_unions.gpkg", driver="GPKG")
    print(f"\n  unions saved -> data_out/sz_barrier_unions.gpkg ({len(unions_gdf)} types)")

print("\nDone.")

=== Barrier Layers Summary ===
  water                 2,161 features [saved]
  waterway              2,061 features [saved]
  railway                  70 features [saved]
  highway_major        42,016 features [saved]
  bridges              22,147 features [saved]
  tunnels               3,447 features [saved]

  unions saved -> data_out/sz_barrier_unions.gpkg (6 types)

Done.
